# Informe de Práctica: Red Bayesiana, Predicción e Implementación de Naive Bayes

## 1. Ejemplo de Aplicación de una Red Bayesiana
**a. Explicación del caso de estudio (Contexto Local)**
Inspirados en el Sistema de Monitoreo y Alerta Temprana de la Calidad del Agua de ETAPA EP en Cuenca, y en investigaciones de la Universidad de Cuenca, se plantea una Red Bayesiana para evaluar el riesgo en la potabilidad del agua en una planta de tratamiento. 
Analizaremos la relación causal entre tres eventos: la presencia de **Lluvias Fuertes (L)**, lo cual puede provocar **Alta Turbiedad (T)** en los ríos, repercutiendo directamente en un **Riesgo en la Potabilidad (R)**.

**b. Variables, Valores y Tablas de Probabilidad:**
*   **Lluvias Fuertes (L):** {Sí, No} *(Variable Raíz)*
    *   `P(L=Sí) = 0.20`, `P(L=No) = 0.80`
*   **Alta Turbiedad (T):** {Sí, No} *(Depende de las Lluvias)*
    *   `P(T=Sí | L=Sí) = 0.85` (Si llueve fuerte, 85% de probabilidad de agua turbia)
    *   `P(T=No | L=Sí) = 0.15`
    *   `P(T=Sí | L=No) = 0.10` (Si no llueve, solo hay un 10% de probabilidad de turbiedad)
    *   `P(T=No | L=No) = 0.90`
*   **Riesgo en Potabilidad (R):** {Alto, Bajo} *(Depende de la Turbiedad)*
    *   `P(R=Alto | T=Sí) = 0.90` (Si hay mucha turbiedad, el riesgo es altísimo)
    *   `P(R=Bajo | T=Sí) = 0.10`
    *   `P(R=Alto | T=No) = 0.05` (Si no hay turbiedad, el riesgo es mínimo)
    *   `P(R=Bajo | T=No) = 0.95`

**Diagrama de Conexiones:**
`[ Lluvias Fuertes (L) ]  ----influye en--->  [ Alta Turbiedad (T) ]  ----influye en--->  [ Riesgo de Potabilidad (R) ]`

**c. Ejemplos de predicción siguiendo la Red Bayesiana:**
1.  **Ejemplo 1 (Midiendo impacto directo):** Si sabemos que el río reporta una Alta Turbiedad (T=Sí), la probabilidad de que exista un Riesgo Alto en la Potabilidad es directamente consultada de la tabla:
    `P(R=Alto | T=Sí) = 0.90 (90%)`
2.  **Ejemplo 2 (Probabilidad marginal de agua turbia):** ¿Cuál es la probabilidad de que cualquier día aleatorio registremos Alta Turbiedad considerando días con y sin lluvia?
    `P(T=Sí) = P(T=Sí | L=Sí) * P(L=Sí)  +  P(T=Sí | L=No) * P(L=No)`
    `P(T=Sí) = (0.85 * 0.20) + (0.10 * 0.80) = 0.17 + 0.08 = 0.25 (25%)`

## 2. Fase de Preparación
Para el componente práctico, cargamos un subset de datos limpios (`mini_water_dataset.csv`) que contiene mediciones numéricas sobre la potabilidad del agua (pH, dureza, sulfatos, etc.), listo para modelado.

In [ ]:
import pandas as pd
from sklearn.naive_bayes import GaussianNB

# 1. Cargar el dataset original (preparado)
df_completo = pd.read_csv('mini_water_dataset.csv')

# Mostrar la tabla final resultante
display(df_completo.head())

## 3. Fase de Modelado (Naive Bayes)
Se particionan las características (`X`) y la variable objetivo (`y`). Posteriormente, empleamos **Gaussian Naive Bayes (GaussianNB)** debido a que las variables independientes son valores numéricos y continuos.

In [ ]:
# Separar las características (X) y la etiqueta que queremos predecir (y)
X = df_completo.drop('Potability', axis=1) 
y = df_completo['Potability']

# Inicializar y entrenar (fit) el modelo GaussianNB
modelo_nb = GaussianNB()
modelo_nb.fit(X, y)
print("¡Modelo Gaussian Naive Bayes entrenado con éxito!")

## 4. Fase de Predicción de Nuevos Samples
Se integró el modelo para discernir la potabilidad con base en características técnicas proporcionando dos muestras totalmente nuevas orientadas a probar la sensibilidad del clasificador:
*   **Sample 1 (Condiciones extremas/contaminadas):** Caracterizada por ser muy ácida (pH: 3.5), altos sólidos disueltos (40000 mg/L) y mucha turbidez (6.5).
*   **Sample 2 (Condiciones controladas/normales):** Muestra neutra (pH: 5.6), con sólidos dentro de rango y turbidez moderada (4.5).

In [ ]:
# Crear 2 muestras de agua completamente nuevas
nuevos_samples = pd.DataFrame({
    'ph': [3.5, 5.6],                       
    'Hardness': [300.0, 190.0],
    'Solids': [40000.0, 13800.0],           
    'Chloramines': [9.0, 6.5],
    'Sulfate': [200.0, 369.0],
    'Conductivity': [600.0, 420.0],
    'Organic_carbon': [25.0, 18.0],
    'Trihalomethanes': [120.0, 80.0],
    'Turbidity': [6.5, 4.5]                 
})

# Realizar la predicción
predicciones = modelo_nb.predict(nuevos_samples)

# Imprimir los resultados
print("Resultados de la Predicción:")
for i, prediccion in enumerate(predicciones):
    estado = "POTABLE" if prediccion == 1 else "NO POTABLE"
    print(f"Sample {i+1} ha sido clasificado como: {estado} (Clase {prediccion})")

## 5. Conclusiones
El enfoque bayesiano permite el desarrollo de sistemas robustos en la ingeniería ambiental y civil. Tal como plantea ETAPA EP en su sistema de alertas tempranas en Cuenca, la monitorización constante y probabilística del estado de los afluentes previene el ingreso de agua intratable a los sistemas de purificación.

El modelo Naive Bayes demostró ser muy eficiente para deducir patrones químicos continuos sin alto poder de cómputo. Aunque la premisa teórica asume que las variables como la conductividad y los cloraminos actúan independiente (cuando no siempre es cierto en el mundo real), la respuesta predictiva fue altamente acertada para los samples propuestos en el programa Python.

El cálculo condicional no sólo ayuda a predecir clasificaciones en base de datos grandes, si no que en análisis gráficos de Redes Bayesianas nos dotan de una línea lógica simple y auditable para entender cómo una perturbación al inicio de la cadena hidrológica penaliza la pureza del recurso hídrico final.

## Referencias
ETAPA EP. (s.f.). *ETAPA EP presenta Sistema de Monitoreo y Alerta Temprana de la Calidad del Agua de Cuenca*. GAD Municipal del Cantón Cuenca. https://www.cuenca.gob.ec/content/etapa-ep-presenta-sistema-de-monitoreo-y-alerta-temprana-de-la-calidad-del-agua-de-cuenca

Universidad de Cuenca. (s.f.). *Documento de investigación sobre Calidad de Agua*. Repositorio Institucional DSpace. https://rest-dspace.ucuenca.edu.ec/server/api/core/bitstreams/b827222a-f2b4-41be-b7c2-ed4b433cb16f/content

Zain, S. (2021). *Water Quality*. Kaggle. https://www.kaggle.com/datasets/sheemazain/water-quality